In [1]:
import re # importa a biblioteca para expressões regulares

In [2]:
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common." # define o texto para análise

In [3]:
pattern = r"Paul [A-Z]\w+" # define o padrão regex

In [4]:
matches = re.finditer(pattern, text) # encontra todas as ocorrencias
for match in matches: # percorre as correspondencias encontradas
    print(match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>


In [6]:
import spacy # importa a biblioteca spacy
from spacy.tokens import Span # importa a classe span

In [7]:
nlp = spacy.blank("en") # cria um modelo em branco em ingles
doc = nlp(text) # processa o texto no modelo
original_ents = list(doc.ents) # converte as entidades do doc em uma lista
mwt_ents = [] # cria uma lista vazia para as novas entidades
for match in re.finditer(pattern, doc.text): # busca o padrão regex dentro do texto do documento
    start, end = match.span() # obtém os índices de início e fim da palavra
    span = doc.char_span(start, end) # converte os índices de caracteres em um objeto span
    print(span) # imprime o span identificado
    if span is not None: # verifica se o span foi criado com sucesso
        mwt_ents.append((span.start, span.end, span.text)) # adiciona os dados do span à lista
for ent in mwt_ents: # itera sobre as entidades capturadas pelo regex
    start, end, text = ent # desempacota os dados da entidade
    span = Span(doc, start, end, label="PERSON") # cria um novo span com o rótulo person
    original_ents.append(span)
doc.ents = original_ents # atualiza as entidades do documento
for ent in doc.ents: # percorre as entidades atualizadas
    print(ent.text, ent.label_)

Paul Newman
Paul Hollywood
Paul Newman PERSON
Paul Hollywood PERSON


In [8]:
print(mwt_ents) # exibe a lista de entidades

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]


In [9]:
from spacy.language import Language # importa o modulo
@Language.component("paul_ner") # registra o novo componente
def paul_ner(doc): # função que processa o documento
    pattern = r"Paul [A-Z]\w+" # define o padrão de busca interno
    original_ents = list(doc.ents) # recupera as entidades já existentes
    mwt_ents = [] # inicia a lista de capturas
    for match in re.finditer(pattern, doc.text): # busca o padrão no texto do documento
        start, end = match.span() # extrai as posições
        span = doc.char_span(start, end) # gera o span
        print(span)
        if span is not None: # garante que o span é válido
            mwt_ents.append((span.start, span.end, span.text)) # guarda os dados da captura
    for ent in mwt_ents: # percorre as capturas feitas
        start, end, text = ent # extrai as informaço
        span = Span(doc, start, end, label="PERSON") # cria o objeto span com rótulo person
        original_ents.append(span) # anexa à lista geral
    doc.ents = original_ents # salva as entidades no documento
    return(doc)

In [10]:
nlp2 = spacy.blank("en") # cria um novo modelo vazio
nlp2.add_pipe("paul_ner") # adiciona o componente customizado

<function __main__.paul_ner(doc)>

In [11]:
doc2 = nlp2(text) # processa o texto com o novo pipeline
print(doc2.ents) # exibe as entidades encontradas

Paul Hollywood
(Paul Hollywood,)


In [14]:
from spacy.util import filter_spans
@Language.component("holly_ner") # registra o componente
def holly_ner(doc): # define a função do componente
    pattern = r"Hollywood" # define o padrão regex para hollywood
    original_ents = list(doc.ents) # lista as entidades atuais
    mwt_ents = [] # lista para novos matches
    for match in re.finditer(pattern, doc.text): # itera sobre os matches de hollywood
        start, end = match.span() # pega os índices de início e fim
        span = doc.char_span(start, end) # cria o span correspondente
        print(span) # mostra o span gerado
        if span is not None: # valida se o span existe
            mwt_ents.append((span.start, span.end, span.text)) # armazena o match
    for ent in mwt_ents: # percorre os matches armazenados
        start, end, text = ent # extrai os dados da entidade
        span = Span(doc, start, end, label="CINEMA") # cria span com rótulo cinema
        original_ents.append(span) # adiciona à lista de entidades
    filtered = filter_spans(original_ents) # filtra spans sobrepostos priorizando os mais longos
    doc.ents = filtered # define as entidades filtradas no documento
    return(doc) # retorna o documento finalizado

In [15]:
nlp3 = spacy.load("en_core_web_sm") # carrega o modelo pre treinado básico
nlp3.add_pipe("holly_ner") # insere o componente de cinema no pipeline

<function __main__.holly_ner(doc)>

In [16]:
doc3 = nlp3(text) # processa o texto original
for ent in doc3.ents: # percorre as entidades do resultado
    print(ent.text, ent.label_) # imprime o texto os rotulos

Hollywood
Paul Hollywood PERSON
